# StayIQ — Module 5: Guest Recovery Copilot Escalation Risk Classifier

This notebook covers training a binary classification model (Gradient Boosting / Random Forest) to predict the **Escalation Risk** of a guest complaint. 

Escalation Risk represents the likelihood that a guest complaint will escalate (e.g. public negative reviews, refund demands, or loyalty churn) if not addressed immediately by staff.

### Pipeline Steps:
1. **Load data**: Read historical complaint records.
2. **Feature Engineering**: Extract features including sentiment scores, negation keyword counts, loyalty tier mappings, past complaint frequencies, and rating-text sentiment mismatches.
3. **Baseline Classifier**: Fit a rule-augmented Gradient Boosting Classifier.
4. **Evaluation**: Compute precision, recall, F1-score, ROC-AUC, and extract feature importance weights.
5. **Model Export**: Save the trained model to `backend/app/models/` for inference.

In [ ]:
# 1. Imports and setup
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve
import joblib

In [ ]:
# 2. Synthesize Training Dataset
# Since actual customer complaint datasets are sensitive, we simulate a representative corpus.
np.random.seed(42)
size = 800

# Features
sentiment_scores = np.random.uniform(0.02, 0.98, size=size)
past_complaints = np.random.choice([0, 1, 2, 3, 4], p=[0.5, 0.25, 0.15, 0.07, 0.03], size=size)
loyalty_tiers = np.random.choice(['Standard', 'Silver', 'Gold', 'Platinum'], p=[0.4, 0.3, 0.2, 0.1], size=size)
negation_counts = np.random.choice([0, 1, 2, 3, 4], p=[0.3, 0.4, 0.2, 0.07, 0.03], size=size)

# Introduce rating mismatch logic (10% of cases)
rating_mismatches = np.random.choice([0, 1], p=[0.9, 0.1], size=size)

# Map loyalty to ordinal values
tier_map = {'Standard': 0, 'Silver': 1, 'Gold': 2, 'Platinum': 3}
loyalty_ordinal = np.array([tier_map[t] for t in loyalty_tiers])

# Risk probability formula with non-linear combinations
base_prob = 1.0 / (1.0 + np.exp(-(
    4.5 * (0.5 - sentiment_scores) 
    + 1.2 * past_complaints 
    + 0.8 * negation_counts 
    + 0.5 * loyalty_ordinal 
    + 1.5 * rating_mismatches 
    - 1.0
)))

# Binarize into escalation target target (1 = Escalates/Critical, 0 = Low Risk)
escalated = (base_prob + np.random.normal(0, 0.1, size=size) > 0.55).astype(int)

df_recovery = pd.DataFrame({
    'sentiment_score': sentiment_scores,
    'past_complaints_count': past_complaints,
    'loyalty_tier': loyalty_tiers,
    'negation_count': negation_counts,
    'rating_mismatch': rating_mismatches,
    'escalated': escalated
})

df_recovery.head()

In [ ]:
# 3. Preprocess Categorical Variables
# Map loyalty tiers numerically for training
df_recovery['loyalty_encoded'] = df_recovery['loyalty_tier'].map(tier_map)

feature_cols = ['sentiment_score', 'past_complaints_count', 'negation_count', 'rating_mismatch', 'loyalty_encoded']
X = df_recovery[feature_cols]
y = df_recovery['escalated']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

In [ ]:
# 4. Fit Gradient Boosting Risk Classifier
gb_classifier = GradientBoostingClassifier(
    n_estimators=80, 
    learning_rate=0.08, 
    max_depth=4, 
    random_state=42
)

print("Fitting classifier...")
gb_classifier.fit(X_train, y_train)
print("Training completed.")

In [ ]:
# 5. Evaluate Metrics
y_pred = gb_classifier.predict(X_test)
y_prob = gb_classifier.predict_proba(X_test)[:, 1]

print("\n=== Escalation Risk Classifier Performance ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Feature Importance Plot
importances = gb_classifier.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(7, 4))
plt.title('Feature Importances in Escalation Risk Model')
plt.barh(range(len(indices)), importances[indices], color='teal', align='center')
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.xlabel('Relative Importance Score')
plt.grid(axis='x', linestyle=':', opacity=0.5)
plt.show()

In [ ]:
# 6. Export Model for Inference
model_export_path = "../backend/app/models/recovery_risk_model.joblib"
joblib.dump(gb_classifier, model_export_path)
print(f"Escalation Risk classifier exported successfully to: {model_export_path}")